In [1]:
# No terminal antes de rodar:
# pip install langgraph

from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import TypedDict, List, Optional
from datetime import datetime
import logging
import json
from pathlib import Path

# Reutiliza o logger da etapa anterior
Path("../logs").mkdir(exist_ok=True)
logging.basicConfig(
    level    = logging.INFO,
    format   = '%(asctime)s | %(levelname)s | %(message)s',
    handlers = [
        logging.FileHandler(
            f"../logs/langgraph_{datetime.now().strftime('%Y%m%d')}.log",
            encoding='utf-8'
        ),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("LangGraph")

print("✅ Imports OK — LangGraph")

✅ Imports OK — LangGraph


In [2]:
# O Estado é o "prontuário digital" que circula entre os nós do grafo
# Cada nó lê e escreve nesse estado — é a memória compartilhada do fluxo

class EstadoPaciente(TypedDict):
    # Dados de entrada
    id_paciente:       str
    nome:              str
    idade:             int
    sintomas:          List[str]
    historico:         List[str]

    # Preenchidos pelos nós do grafo
    exames_pendentes:  List[str]
    nivel_risco:       str        # "baixo", "moderado", "alto", "critico"
    conduta_sugerida:  str
    alertas:           List[str]
    relatorio_final:   str

    # Controle de fluxo
    requer_alerta:     bool
    etapas_executadas: List[str]


# Paciente de exemplo para testar o fluxo
paciente_exemplo = EstadoPaciente(
    id_paciente       = "PAC-2024-001",
    nome              = "Maria Silva",
    idade             = 52,
    sintomas          = [
        "nódulo palpável na mama direita",
        "dor ocasional no local",
        "retração mamilar recente"
    ],
    historico         = [
        "histórico familiar de câncer de mama (mãe)",
        "menarca aos 11 anos",
        "nulípara",
        "uso de terapia hormonal por 5 anos"
    ],
    exames_pendentes  = [],
    nivel_risco       = "",
    conduta_sugerida  = "",
    alertas           = [],
    relatorio_final   = "",
    requer_alerta     = False,
    etapas_executadas = [],
)

print("✅ Estado do paciente definido!")
print(f"   Paciente : {paciente_exemplo['nome']}, {paciente_exemplo['idade']} anos")
print(f"   Sintomas : {paciente_exemplo['sintomas']}")

✅ Estado do paciente definido!
   Paciente : Maria Silva, 52 anos
   Sintomas : ['nódulo palpável na mama direita', 'dor ocasional no local', 'retração mamilar recente']


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()  # carrega o .env automaticamente

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY não encontrada!\n"
        "   Crie um arquivo .env na raiz do projeto com:\n"
        "   ANTHROPIC_API_KEY=sua-chave-aqui"
    )

print("✅ Chave carregada com segurança via variável de ambiente")

llm = ChatAnthropic(
    model       = "claude-sonnet-4-6",
    api_key     = ANTHROPIC_API_KEY,
    temperature = 0.2,
    max_tokens  = 1024,
)

print("✅ LLM configurada!")

✅ Chave carregada com segurança via variável de ambiente
✅ LLM configurada!


In [4]:
# Cada função é um NÓ do grafo — recebe o estado e retorna as atualizações

# ── NÓ 1: Triagem inicial ────────────────────────────────────────
def no_triagem(estado: EstadoPaciente) -> dict:
    """
    Analisa os sintomas e determina o nível de risco inicial.
    """
    logger.info(f"[{estado['id_paciente']}] Nó: TRIAGEM")

    prompt = ChatPromptTemplate.from_template("""
Você é um sistema de triagem médica. Analise os dados do paciente
e classifique o nível de risco.

PACIENTE: {nome}, {idade} anos
SINTOMAS: {sintomas}
HISTÓRICO: {historico}

Classifique o nível de risco como:
- "baixo": sintomas inespecíficos, sem fatores de risco
- "moderado": alguns fatores de risco presentes
- "alto": múltiplos fatores de risco ou sintomas sugestivos
- "critico": sintomas graves que exigem atenção imediata

Responda APENAS com um JSON no formato:
{{"nivel_risco": "alto", "justificativa": "..."}}
""")

    chain  = prompt | llm | StrOutputParser()
    result = chain.invoke({
        "nome":      estado["nome"],
        "idade":     estado["idade"],
        "sintomas":  ", ".join(estado["sintomas"]),
        "historico": ", ".join(estado["historico"]),
    })

    try:
        # Remove possíveis backticks do JSON
        result_clean = result.strip().strip("```json").strip("```").strip()
        dados        = json.loads(result_clean)
        nivel_risco  = dados.get("nivel_risco", "moderado")
    except Exception:
        nivel_risco  = "moderado"

    logger.info(f"[{estado['id_paciente']}] Risco: {nivel_risco}")

    return {
        "nivel_risco":       nivel_risco,
        "etapas_executadas": estado["etapas_executadas"] + ["triagem"],
    }


# ── NÓ 2: Verificação de exames ──────────────────────────────────
def no_exames(estado: EstadoPaciente) -> dict:
    """
    Sugere exames complementares com base nos sintomas e risco.
    """
    logger.info(f"[{estado['id_paciente']}] Nó: EXAMES")

    prompt = ChatPromptTemplate.from_template("""
Você é um assistente médico. Com base nos dados do paciente,
sugira os exames complementares necessários.

PACIENTE: {nome}, {idade} anos
SINTOMAS: {sintomas}
NÍVEL DE RISCO: {nivel_risco}

Liste os exames em ordem de prioridade.
Responda APENAS com um JSON no formato:
{{"exames": ["exame1", "exame2", "exame3"]}}
""")

    chain  = prompt | llm | StrOutputParser()
    result = chain.invoke({
        "nome":        estado["nome"],
        "idade":       estado["idade"],
        "sintomas":    ", ".join(estado["sintomas"]),
        "nivel_risco": estado["nivel_risco"],
    })

    try:
        result_clean    = result.strip().strip("```json").strip("```").strip()
        dados           = json.loads(result_clean)
        exames          = dados.get("exames", [])
    except Exception:
        exames = ["Mamografia bilateral", "Ultrassonografia mamária"]

    logger.info(f"[{estado['id_paciente']}] Exames sugeridos: {exames}")

    return {
        "exames_pendentes":  exames,
        "etapas_executadas": estado["etapas_executadas"] + ["exames"],
    }


# ── NÓ 3: Sugestão de conduta ────────────────────────────────────
def no_conduta(estado: EstadoPaciente) -> dict:
    """
    Sugere a conduta clínica baseada em todos os dados coletados.
    """
    logger.info(f"[{estado['id_paciente']}] Nó: CONDUTA")

    prompt = ChatPromptTemplate.from_template("""
Você é um assistente médico de apoio clínico. Sugira uma conduta
baseada nos dados do paciente. NUNCA prescreva medicamentos diretamente.

PACIENTE: {nome}, {idade} anos
SINTOMAS: {sintomas}
HISTÓRICO: {historico}
NÍVEL DE RISCO: {nivel_risco}
EXAMES SUGERIDOS: {exames}

Forneça uma conduta clínica estruturada em português brasileiro.
Inclua: encaminhamentos sugeridos, urgência do atendimento e
orientações gerais. Lembre que o médico tem a palavra final.
""")

    chain   = prompt | llm | StrOutputParser()
    conduta = chain.invoke({
        "nome":        estado["nome"],
        "idade":       estado["idade"],
        "sintomas":    ", ".join(estado["sintomas"]),
        "historico":   ", ".join(estado["historico"]),
        "nivel_risco": estado["nivel_risco"],
        "exames":      ", ".join(estado["exames_pendentes"]),
    })

    logger.info(f"[{estado['id_paciente']}] Conduta gerada")

    return {
        "conduta_sugerida":  conduta,
        "etapas_executadas": estado["etapas_executadas"] + ["conduta"],
    }


# ── NÓ 4: Sistema de alertas ─────────────────────────────────────
def no_alerta(estado: EstadoPaciente) -> dict:
    """
    Emite alertas para a equipe médica quando necessário.
    Ativado apenas para riscos alto ou crítico.
    """
    logger.info(f"[{estado['id_paciente']}] Nó: ALERTA")

    alertas = []

    if estado["nivel_risco"] in ["alto", "critico"]:
        alertas.append(
            f"🚨 ALERTA [{estado['nivel_risco'].upper()}]: "
            f"Paciente {estado['nome']} requer atenção prioritária."
        )

    if estado["nivel_risco"] == "critico":
        alertas.append(
            "🔴 ATENÇÃO IMEDIATA: Encaminhar para avaliação especializada urgente!"
        )

    if any("mama" in s.lower() for s in estado["sintomas"]):
        alertas.append(
            "⚠️ Sintomas mamários detectados: Avaliação mastológica recomendada."
        )

    for alerta in alertas:
        logger.warning(f"[{estado['id_paciente']}] {alerta}")

    return {
        "alertas":           alertas,
        "requer_alerta":     len(alertas) > 0,
        "etapas_executadas": estado["etapas_executadas"] + ["alerta"],
    }


# ── NÓ 5: Relatório final ────────────────────────────────────────
def no_relatorio(estado: EstadoPaciente) -> dict:
    """
    Consolida todas as informações em um relatório final estruturado.
    """
    logger.info(f"[{estado['id_paciente']}] Nó: RELATÓRIO")

    alertas_texto = "\n".join(estado["alertas"]) if estado["alertas"] else "Nenhum alerta."

    relatorio = f"""
╔══════════════════════════════════════════════════════════════╗
   RELATÓRIO CLÍNICO — ASSISTENTE MÉDICO IA
   Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}
╚══════════════════════════════════════════════════════════════╝

👤 PACIENTE
   ID      : {estado['id_paciente']}
   Nome    : {estado['nome']}
   Idade   : {estado['idade']} anos

🔴 NÍVEL DE RISCO: {estado['nivel_risco'].upper()}

🩺 SINTOMAS RELATADOS
{chr(10).join(f'   • {s}' for s in estado['sintomas'])}

📋 HISTÓRICO CLÍNICO
{chr(10).join(f'   • {h}' for h in estado['historico'])}

🔬 EXAMES SUGERIDOS
{chr(10).join(f'   • {e}' for e in estado['exames_pendentes'])}

📌 CONDUTA SUGERIDA
{estado['conduta_sugerida']}

⚠️  ALERTAS
{alertas_texto}

🔄 ETAPAS EXECUTADAS: {' → '.join(estado['etapas_executadas'])}

══════════════════════════════════════════════════════════════
⚕️  AVISO: Este relatório é de apoio clínico.
    O médico responsável tem a palavra final.
══════════════════════════════════════════════════════════════
"""

    logger.info(f"[{estado['id_paciente']}] Relatório gerado com sucesso")

    return {
        "relatorio_final":   relatorio,
        "etapas_executadas": estado["etapas_executadas"] + ["relatorio"],
    }


print("✅ Todos os nós definidos!")
print("   → no_triagem | no_exames | no_conduta | no_alerta | no_relatorio")

✅ Todos os nós definidos!
   → no_triagem | no_exames | no_conduta | no_alerta | no_relatorio


In [5]:
# Monta o fluxo de decisão como um grafo direcionado
# Cada aresta define para onde o fluxo vai após cada nó

grafo = StateGraph(EstadoPaciente)

# Adiciona os nós
grafo.add_node("triagem",  no_triagem)
grafo.add_node("exames",   no_exames)
grafo.add_node("conduta",  no_conduta)
grafo.add_node("alerta",   no_alerta)
grafo.add_node("relatorio",no_relatorio)

# Define o ponto de entrada
grafo.set_entry_point("triagem")

# Define as arestas (fluxo linear com desvio condicional)
grafo.add_edge("triagem", "exames")
grafo.add_edge("exames",  "conduta")
grafo.add_edge("conduta", "alerta")
grafo.add_edge("alerta",  "relatorio")
grafo.add_edge("relatorio", END)

# Compila o grafo
app = grafo.compile()

print("✅ Grafo LangGraph compilado!")
print("\n   Fluxo:")
print("   triagem → exames → conduta → alerta → relatório → FIM")

✅ Grafo LangGraph compilado!

   Fluxo:
   triagem → exames → conduta → alerta → relatório → FIM


In [6]:
print("🏥 EXECUTANDO FLUXO AUTOMATIZADO DE DECISÃO CLÍNICA\n")
print("="*65)
print(f"Paciente: {paciente_exemplo['nome']}, {paciente_exemplo['idade']} anos")
print(f"Sintomas: {paciente_exemplo['sintomas']}")
print("="*65)

# Executa o grafo com o paciente de exemplo
resultado_final = app.invoke(paciente_exemplo)

# Exibe o relatório completo
print(resultado_final["relatorio_final"])

# Exibe métricas
print(f"\n📊 MÉTRICAS DA EXECUÇÃO")
print(f"   Etapas executadas : {' → '.join(resultado_final['etapas_executadas'])}")
print(f"   Nível de risco    : {resultado_final['nivel_risco'].upper()}")
print(f"   Exames sugeridos  : {len(resultado_final['exames_pendentes'])}")
print(f"   Alertas emitidos  : {len(resultado_final['alertas'])}")

2026-09-15 13:43:11,189 | INFO | [PAC-2024-001] Nó: TRIAGEM


🏥 EXECUTANDO FLUXO AUTOMATIZADO DE DECISÃO CLÍNICA

Paciente: Maria Silva, 52 anos
Sintomas: ['nódulo palpável na mama direita', 'dor ocasional no local', 'retração mamilar recente']


2026-09-15 13:43:16,924 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-15 13:43:16,941 | INFO | [PAC-2024-001] Risco: alto
2026-09-15 13:43:16,942 | INFO | [PAC-2024-001] Nó: EXAMES
2026-09-15 13:43:25,236 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-15 13:43:25,238 | INFO | [PAC-2024-001] Exames sugeridos: ['Mamografia bilateral', 'Ultrassonografia mamária']
2026-09-15 13:43:25,239 | INFO | [PAC-2024-001] Nó: CONDUTA
2026-09-15 13:43:44,582 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-15 13:43:44,584 | INFO | [PAC-2024-001] Conduta gerada
2026-09-15 13:43:44,584 | INFO | [PAC-2024-001] Nó: ALERTA
2026-09-15 13:43:44,584 | WARNING | [PAC-2024-001] 🚨 ALERTA [ALTO]: Paciente Maria Silva requer atenção prioritária.
2026-09-15 13:43:44,585 | WARNING | [PAC-2024-001] ⚠️ Sintomas mamários detectados: Avaliação mastológica recomendada.
2026-09-15 13:43:44,


╔══════════════════════════════════════════════════════════════╗
   RELATÓRIO CLÍNICO — ASSISTENTE MÉDICO IA
   Gerado em: 15/09/2026 13:43:44
╚══════════════════════════════════════════════════════════════╝

👤 PACIENTE
   ID      : PAC-2024-001
   Nome    : Maria Silva
   Idade   : 52 anos

🔴 NÍVEL DE RISCO: ALTO

🩺 SINTOMAS RELATADOS
   • nódulo palpável na mama direita
   • dor ocasional no local
   • retração mamilar recente

📋 HISTÓRICO CLÍNICO
   • histórico familiar de câncer de mama (mãe)
   • menarca aos 11 anos
   • nulípara
   • uso de terapia hormonal por 5 anos

🔬 EXAMES SUGERIDOS
   • Mamografia bilateral
   • Ultrassonografia mamária

📌 CONDUTA SUGERIDA
# Conduta Clínica Sugerida — Maria Silva, 52 anos

---

> ⚠️ **AVISO IMPORTANTE:** Este documento é um **apoio clínico estruturado** e **não substitui** a avaliação médica presencial. O médico responsável tem autonomia total sobre as decisões diagnósticas e terapêuticas.

---

## 🔴 Nível de Urgência: **ALTO — Atendimento

In [7]:
# Testa o fluxo com um caso de baixo risco para comparar
paciente_baixo_risco = EstadoPaciente(
    id_paciente       = "PAC-2024-002",
    nome              = "Ana Souza",
    idade             = 28,
    sintomas          = ["dor de cabeça ocasional", "cansaço leve"],
    historico         = ["sem histórico familiar relevante", "saudável"],
    exames_pendentes  = [],
    nivel_risco       = "",
    conduta_sugerida  = "",
    alertas           = [],
    relatorio_final   = "",
    requer_alerta     = False,
    etapas_executadas = [],
)

print("🏥 SEGUNDO PACIENTE — Baixo risco\n")
print("="*65)

resultado_2 = app.invoke(paciente_baixo_risco)
print(resultado_2["relatorio_final"])
print(f"   Alertas emitidos: {len(resultado_2['alertas'])}")
print(f"   Nível de risco  : {resultado_2['nivel_risco'].upper()}")

2026-09-15 13:44:37,812 | INFO | [PAC-2024-002] Nó: TRIAGEM


🏥 SEGUNDO PACIENTE — Baixo risco



2026-09-15 13:44:41,957 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-15 13:44:41,959 | INFO | [PAC-2024-002] Risco: baixo
2026-09-15 13:44:41,959 | INFO | [PAC-2024-002] Nó: EXAMES
2026-09-15 13:44:44,465 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-15 13:44:44,467 | INFO | [PAC-2024-002] Exames sugeridos: ['Hemograma completo', 'Dosagem de ferritina e ferro sérico', 'Dosagem de TSH (hormônio estimulante da tireoide)', 'Glicemia em jejum', 'Dosagem de vitamina B12 e ácido fólico', 'Aferição de pressão arterial']
2026-09-15 13:44:44,467 | INFO | [PAC-2024-002] Nó: CONDUTA
2026-09-15 13:45:04,586 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-15 13:45:04,587 | INFO | [PAC-2024-002] Conduta gerada
2026-09-15 13:45:04,588 | INFO | [PAC-2024-002] Nó: ALERTA
2026-09-15 13:45:04,588 | INFO | [PAC-2024-002] Nó: RELATÓRIO
2026-09-15 13:45:04,589 | INFO | [PA


╔══════════════════════════════════════════════════════════════╗
   RELATÓRIO CLÍNICO — ASSISTENTE MÉDICO IA
   Gerado em: 15/09/2026 13:45:04
╚══════════════════════════════════════════════════════════════╝

👤 PACIENTE
   ID      : PAC-2024-002
   Nome    : Ana Souza
   Idade   : 28 anos

🔴 NÍVEL DE RISCO: BAIXO

🩺 SINTOMAS RELATADOS
   • dor de cabeça ocasional
   • cansaço leve

📋 HISTÓRICO CLÍNICO
   • sem histórico familiar relevante
   • saudável

🔬 EXAMES SUGERIDOS
   • Hemograma completo
   • Dosagem de ferritina e ferro sérico
   • Dosagem de TSH (hormônio estimulante da tireoide)
   • Glicemia em jejum
   • Dosagem de vitamina B12 e ácido fólico
   • Aferição de pressão arterial

📌 CONDUTA SUGERIDA
# 📋 Conduta Clínica Sugerida

**Paciente:** Ana Souza, 28 anos
**Data de análise:** 29/05/2025
**Nível de risco:** 🟢 Baixo

---

## 🔍 Impressão Clínica Geral

O quadro de **cefaleia ocasional associada a fadiga leve** em paciente jovem, saudável e sem histórico familiar relevante 

## 💬 Discussão Crítica — Fase 3

### Fine-tuning com QLoRA
O fine-tuning foi realizado no modelo LLaMA 3.2 1B Instruct utilizando
QLoRA — técnica que congela os pesos originais e treina apenas adaptadores
de baixo rank (r=16) em camadas de atenção e feed-forward. O treinamento
usou 900 exemplos (90% do dataset) por 3 épocas, atingindo loss final de
1.3427 — resultado considerado bom para um dataset de 1000 exemplos,
indicando aprendizado sem overfitting.

A escolha do LLaMA 3.2 1B foi estratégica: é o menor modelo da família
LLaMA 3.2 com suporte a instruções, cabendo em GPUs de 16GB (T4 do Colab)
com quantização 4-bit. Modelos maiores (7B, 13B) produziriam respostas de
maior qualidade, mas exigiriam hardware mais robusto.

### Pipeline RAG com LangChain
O pipeline RAG combina três componentes. O Vector Store FAISS indexa os
documentos médicos como vetores usando embeddings BioBERT — modelo
especializado em textos biomédicos que captura semântica clínica melhor
do que embeddings genéricos. O Retriever recupera os 4 documentos mais
relevantes por similaridade cosseno. O LLM Claude claude-sonnet-4-6
com temperatura 0.2 gera respostas conservadoras e estruturadas.

Uma limitação observada foi que com 1000 exemplos a recuperação nem
sempre retorna documentos diretamente relacionados à pergunta — o Claude
compensa com conhecimento próprio mas cita fontes genéricas. Em produção,
uma base com dezenas de milhares de documentos especializados resolveria
esse problema.

### Fluxo automatizado com LangGraph
O LangGraph implementa um grafo de decisão clínica com 5 nós sequenciais:
triagem → exames → conduta → alerta → relatório. O estado compartilhado
(EstadoPaciente) funciona como um prontuário digital que cada nó lê e
enriquece. O sistema de alertas é ativado condicionalmente com base no
nível de risco — apenas pacientes de risco alto ou crítico recebem alertas,
evitando sobrecarga de notificações.

### Segurança e validação
Três camadas de segurança foram implementadas. A validação de tópicos
bloqueados impede solicitações de prescrição direta via regex. O logging
estruturado registra todas as consultas com timestamp, ID do profissional
e fontes utilizadas, garantindo rastreabilidade completa. A explainability
é garantida pela citação explícita das fontes recuperadas pelo FAISS em
cada resposta.

### Limitações e próximos passos
O sistema não possui autenticação de usuários — em produção seria
necessário integrar com sistemas de identidade hospitalar. O modelo
fine-tunado responde em inglês por ter sido treinado com dados em inglês
(PubMedQA e MedQuAD) — o system prompt em português contorna isso mas
não é ideal. A solução ideal seria fine-tuning com dados clínicos em
português, como prontuários anonimizados de hospitais brasileiros.

O modelo é uma ferramenta de apoio clínico — nunca de substituição
ao médico. O profissional de saúde tem sempre a palavra final.

In [8]:
print("=" * 60)
print("  ✅ TECH CHALLENGE FASE 3 — CONCLUÍDO!")
print("=" * 60)

entregas = [
    ("Fine-tuning QLoRA — LLaMA 3.2 1B",         "✅"),
    ("Dataset: PubMedQA + MedQuAD (1000 ex.)",    "✅"),
    ("Loss final: 1.3427",                         "✅"),
    ("Pipeline RAG com LCEL + FAISS + Claude",     "✅"),
    ("Embeddings BioBERT especializados",          "✅"),
    ("Validação de tópicos bloqueados",            "✅"),
    ("Logging estruturado com auditoria",          "✅"),
    ("Citação de fontes (explainability)",         "✅"),
    ("LangGraph — fluxo de decisão clínica",       "✅"),
    ("5 nós: triagem→exames→conduta→alerta→rel.",  "✅"),
    ("Relatório clínico automatizado",             "✅"),
    ("Discussão crítica em Markdown",              "✅"),
]

print("\n📋 Entregas técnicas:")
for item, status in entregas:
    print(f"   {status} {item}")

print(f"\n📁 Notebooks gerados:")
notebooks = [
    "01_preparacao_dados.ipynb",
    "02_fine_tuning.ipynb      (executado no Colab)",
    "03_rag_langchain.ipynb",
    "04_langgraph.ipynb",
]
for nb in notebooks:
    print(f"   📓 {nb}")

print(f"\n🎉 Projeto Fase 3 completo!")
print(f"   Fine-tuning + RAG + LangChain + LangGraph")
print(f"   Tudo funcionando fim a fim!")

  ✅ TECH CHALLENGE FASE 3 — CONCLUÍDO!

📋 Entregas técnicas:
   ✅ Fine-tuning QLoRA — LLaMA 3.2 1B
   ✅ Dataset: PubMedQA + MedQuAD (1000 ex.)
   ✅ Loss final: 1.3427
   ✅ Pipeline RAG com LCEL + FAISS + Claude
   ✅ Embeddings BioBERT especializados
   ✅ Validação de tópicos bloqueados
   ✅ Logging estruturado com auditoria
   ✅ Citação de fontes (explainability)
   ✅ LangGraph — fluxo de decisão clínica
   ✅ 5 nós: triagem→exames→conduta→alerta→rel.
   ✅ Relatório clínico automatizado
   ✅ Discussão crítica em Markdown

📁 Notebooks gerados:
   📓 01_preparacao_dados.ipynb
   📓 02_fine_tuning.ipynb      (executado no Colab)
   📓 03_rag_langchain.ipynb
   📓 04_langgraph.ipynb

🎉 Projeto Fase 3 completo!
   Fine-tuning + RAG + LangChain + LangGraph
   Tudo funcionando fim a fim!
